In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, when, isnan, avg, min, max, stddev, round,
    regexp_replace, regexp_extract,
    hour, dayofmonth, month, year, to_date
)
from pyspark.sql.types import DoubleType, FloatType, IntegerType, LongType
import matplotlib.pyplot as plt
import pandas as pd
import unicodedata
import re

In [ ]:
spark = (
    SparkSession.builder
    .appName("EDA_Weather_Surface_Brazil_Southeast")
    .getOrCreate()
)

spark

In [ ]:
parquet_path = "/home/jovyan/work/data/processed/weather_southeast_parquet"

df = spark.read.parquet(parquet_path)

df.show(5, truncate=False)
df.printSchema()

In [ ]:
total_linhas = df.count()
total_colunas = len(df.columns)

print(f"Total de linhas: {total_linhas}")
print(f"Total de colunas: {total_colunas}")

In [ ]:
df.columns

In [ ]:
def normalizar_nome_coluna(nome):
    nome = nome.strip()
    nome = unicodedata.normalize("NFKD", nome)
    nome = nome.encode("ASCII", "ignore").decode("utf-8")
    nome = nome.lower()
    nome = re.sub(r"[^a-z0-9]+", "_", nome)
    nome = re.sub(r"_+", "_", nome)
    nome = nome.strip("_")
    return nome

colunas_normalizadas = [normalizar_nome_coluna(c) for c in df.columns]

df = df.toDF(*colunas_normalizadas)

df.columns

In [ ]:
df.show(10, truncate=False)
df.printSchema()

In [ ]:
df_weather = (
    df
    .withColumnRenamed("data", "data")
    .withColumnRenamed("hora", "hora")
    .withColumnRenamed("precipitacao_total_horario_mm", "precipitacao")
    .withColumnRenamed("pressao_atmosferica_ao_nivel_da_estacao_horaria_mb", "pressao")
    .withColumnRenamed("pressao_atmosferica_max_na_hora_ant_aut_mb", "pressao_maxima")
    .withColumnRenamed("pressao_atmosferica_min_na_hora_ant_aut_mb", "pressao_minima")
    .withColumnRenamed("radiacao_global_kj_m2", "radiacao")
    .withColumnRenamed("temperatura_do_ar_bulbo_seco_horaria_c", "temperatura")
    .withColumnRenamed("temperatura_do_ponto_de_orvalho_c", "temperatura_orvalho")
    .withColumnRenamed("temperatura_maxima_na_hora_ant_aut_c", "temperatura_maxima")
    .withColumnRenamed("temperatura_minima_na_hora_ant_aut_c", "temperatura_minima")
    .withColumnRenamed("temperatura_orvalho_max_na_hora_ant_aut_c", "temperatura_orvalho_maxima")
    .withColumnRenamed("temperatura_orvalho_min_na_hora_ant_aut_c", "temperatura_orvalho_minima")
    .withColumnRenamed("umidade_rel_max_na_hora_ant_aut", "umidade_maxima")
    .withColumnRenamed("umidade_rel_min_na_hora_ant_aut", "umidade_minima")
    .withColumnRenamed("umidade_relativa_do_ar_horaria", "umidade")
    .withColumnRenamed("vento_direcao_horaria_gr_gr", "direcao_vento")
    .withColumnRenamed("vento_rajada_maxima_m_s", "rajada_vento")
    .withColumnRenamed("vento_velocidade_horaria_m_s", "velocidade_vento")
)

df_weather.printSchema()
df_weather.show(5, truncate=False)

In [ ]:
colunas_uteis = [
    "data",
    "hora",
    "temperatura",
    "temperatura_orvalho",
    "temperatura_maxima",
    "temperatura_minima",
    "umidade",
    "umidade_maxima",
    "umidade_minima",
    "pressao",
    "pressao_maxima",
    "pressao_minima",
    "precipitacao",
    "velocidade_vento",
    "rajada_vento",
    "direcao_vento",
    "radiacao",
    "region",
    "state",
    "station",
    "station_code"
]

colunas_uteis = [c for c in colunas_uteis if c in df_weather.columns]

df_weather = df_weather.select(*colunas_uteis)

df_weather.show(10, truncate=False)
df_weather.printSchema()

In [ ]:
colunas_numericas = [
    "temperatura",
    "temperatura_orvalho",
    "temperatura_maxima",
    "temperatura_minima",
    "umidade",
    "umidade_maxima",
    "umidade_minima",
    "pressao",
    "pressao_maxima",
    "pressao_minima",
    "precipitacao",
    "velocidade_vento",
    "rajada_vento",
    "direcao_vento",
    "radiacao"
]

colunas_numericas = [c for c in colunas_numericas if c in df_weather.columns]

for c in colunas_numericas:
    df_weather = df_weather.withColumn(
        c,
        regexp_replace(col(c).cast("string"), ",", ".").cast("double")
    )

df_weather.printSchema()
df_weather.show(10, truncate=False)

In [ ]:
for c in colunas_numericas:
    df_weather = df_weather.withColumn(
        c,
        when(col(c) <= -999, None).otherwise(col(c))
    )

df_weather.show(10, truncate=False)

In [ ]:
colunas_numericas_spark = [
    field.name
    for field in df_weather.schema.fields
    if isinstance(field.dataType, (DoubleType, FloatType, IntegerType, LongType))
]

expressoes_nulos = []

for c in df_weather.columns:
    if c in colunas_numericas_spark:
        expressoes_nulos.append(
            count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
        )
    else:
        expressoes_nulos.append(
            count(when(col(c).isNull(), c)).alias(c)
        )

df_weather.select(expressoes_nulos).show(truncate=False)

In [ ]:
total_registros = df_weather.count()

expressoes_percentual_nulos = []

for c in df_weather.columns:
    if c in colunas_numericas_spark:
        expressoes_percentual_nulos.append(
            round(
                (count(when(col(c).isNull() | isnan(col(c)), c)) / total_registros) * 100,
                2
            ).alias(c)
        )
    else:
        expressoes_percentual_nulos.append(
            round(
                (count(when(col(c).isNull(), c)) / total_registros) * 100,
                2
            ).alias(c)
        )

df_weather.select(expressoes_percentual_nulos).show(truncate=False)

In [ ]:
df_weather.describe().show(truncate=False)

In [ ]:
df_weather.select([
    round(avg(c), 2).alias(f"media_{c}") for c in colunas_numericas
]).show(truncate=False)

df_weather.select([
    round(min(c), 2).alias(f"min_{c}") for c in colunas_numericas
]).show(truncate=False)

df_weather.select([
    round(max(c), 2).alias(f"max_{c}") for c in colunas_numericas
]).show(truncate=False)

df_weather.select([
    round(stddev(c), 2).alias(f"desvio_padrao_{c}") for c in colunas_numericas
]).show(truncate=False)

In [ ]:
df_sample = (
    df_weather
    .dropna(subset=["temperatura"])
    .sample(fraction=0.02, seed=42)
)

pdf_sample = df_sample.toPandas()

pdf_sample.head()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(pdf_sample["temperatura"].dropna(), bins=40)
plt.title("Distribuição da Temperatura")
plt.xlabel("Temperatura")
plt.ylabel("Frequência")
plt.show()

In [ ]:
for c in colunas_numericas:
    if c in pdf_sample.columns:
        plt.figure(figsize=(10, 5))
        plt.hist(pdf_sample[c].dropna(), bins=40)
        plt.title(f"Distribuição de {c}")
        plt.xlabel(c)
        plt.ylabel("Frequência")
        plt.show()

In [ ]:
for c in colunas_numericas:
    if c in pdf_sample.columns:
        plt.figure(figsize=(10, 4))
        plt.boxplot(pdf_sample[c].dropna(), vert=False)
        plt.title(f"Boxplot de {c}")
        plt.xlabel(c)
        plt.show()

In [ ]:
df_weather_clean = df_weather

if "temperatura" in df_weather_clean.columns:
    df_weather_clean = df_weather_clean.filter(
        (col("temperatura").isNull()) |
        ((col("temperatura") >= -10) & (col("temperatura") <= 50))
    )

if "temperatura_orvalho" in df_weather_clean.columns:
    df_weather_clean = df_weather_clean.filter(
        (col("temperatura_orvalho").isNull()) |
        ((col("temperatura_orvalho") >= -20) & (col("temperatura_orvalho") <= 40))
    )

if "umidade" in df_weather_clean.columns:
    df_weather_clean = df_weather_clean.filter(
        (col("umidade").isNull()) |
        ((col("umidade") >= 0) & (col("umidade") <= 100))
    )

if "precipitacao" in df_weather_clean.columns:
    df_weather_clean = df_weather_clean.filter(
        (col("precipitacao").isNull()) |
        (col("precipitacao") >= 0)
    )

if "velocidade_vento" in df_weather_clean.columns:
    df_weather_clean = df_weather_clean.filter(
        (col("velocidade_vento").isNull()) |
        (col("velocidade_vento") >= 0)
    )

if "rajada_vento" in df_weather_clean.columns:
    df_weather_clean = df_weather_clean.filter(
        (col("rajada_vento").isNull()) |
        (col("rajada_vento") >= 0)
    )

print("Linhas antes da limpeza:", df_weather.count())
print("Linhas depois da limpeza:", df_weather_clean.count())

In [ ]:
df_weather_clean = df_weather_clean.withColumn(
    "categoria_temperatura",
    when(col("temperatura") < 15, "Frio")
    .when(col("temperatura") < 25, "Ameno")
    .when(col("temperatura") < 35, "Quente")
    .otherwise("Muito quente")
)

df_weather_clean.select("temperatura", "categoria_temperatura").show(20)

In [ ]:
df_weather_clean.groupBy("categoria_temperatura") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

In [ ]:
if "precipitacao" in df_weather_clean.columns:
    df_weather_clean = df_weather_clean.withColumn(
        "ocorrencia_chuva",
        when(col("precipitacao") > 0, "Com chuva")
        .otherwise("Sem chuva")
    )

    df_weather_clean.groupBy("ocorrencia_chuva").count().show()

In [ ]:
df_weather_clean.groupBy("categoria_temperatura").agg(
    round(avg("temperatura"), 2).alias("media_temperatura"),
    round(avg("umidade"), 2).alias("media_umidade"),
    round(avg("pressao"), 2).alias("media_pressao"),
    round(avg("precipitacao"), 2).alias("media_precipitacao"),
    round(avg("velocidade_vento"), 2).alias("media_velocidade_vento")
).orderBy("media_temperatura").show(truncate=False)

In [ ]:
if "ocorrencia_chuva" in df_weather_clean.columns:
    df_weather_clean.groupBy("ocorrencia_chuva").agg(
        count("*").alias("total_registros"),
        round(avg("temperatura"), 2).alias("media_temperatura"),
        round(avg("umidade"), 2).alias("media_umidade"),
        round(avg("pressao"), 2).alias("media_pressao"),
        round(avg("precipitacao"), 2).alias("media_precipitacao")
    ).show(truncate=False)

In [ ]:
correlacoes = []

for c in colunas_numericas:
    if c != "temperatura" and c in df_weather_clean.columns:
        df_corr_temp = df_weather_clean.select("temperatura", c).dropna()
        
        if df_corr_temp.count() > 0:
            corr = df_corr_temp.stat.corr("temperatura", c)
            correlacoes.append((c, corr))

df_corr = spark.createDataFrame(correlacoes, ["variavel", "correlacao_com_temperatura"])

df_corr.orderBy(col("correlacao_com_temperatura").desc()).show(truncate=False)

In [ ]:
pdf_corr = df_corr.toPandas().sort_values("correlacao_com_temperatura", ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(pdf_corr["variavel"], pdf_corr["correlacao_com_temperatura"])
plt.title("Correlação das Variáveis com a Temperatura")
plt.xlabel("Correlação")
plt.ylabel("Variável")
plt.show()

In [ ]:
df_temporal = df_weather_clean

if "data" in df_temporal.columns:
    df_temporal = df_temporal.withColumn("data_formatada", to_date(col("data")))

    df_temporal = (
        df_temporal
        .withColumn("ano", year(col("data_formatada")))
        .withColumn("mes", month(col("data_formatada")))
        .withColumn("dia", dayofmonth(col("data_formatada")))
    )

if "hora" in df_temporal.columns:
    df_temporal = df_temporal.withColumn(
        "hora_num",
        regexp_extract(col("hora").cast("string"), r"(\d{2})", 1).cast("int")
    )

df_temporal.select("data", "hora", "ano", "mes", "dia", "hora_num").show(20)


# Se ano, mes e dia ficarem nulos, rode esta alternativa:


# df_temporal = df_weather_clean

# if "data" in df_temporal.columns:
#     df_temporal = df_temporal.withColumn("data_formatada", to_date(col("data"), "dd/MM/yyyy"))

#     df_temporal = (
#         df_temporal
#         .withColumn("ano", year(col("data_formatada")))
#         .withColumn("mes", month(col("data_formatada")))
#         .withColumn("dia", dayofmonth(col("data_formatada")))
#     )

# if "hora" in df_temporal.columns:
#     df_temporal = df_temporal.withColumn(
#         "hora_num",
#         regexp_extract(col("hora").cast("string"), r"(\d{2})", 1).cast("int")
#     )

# df_temporal.select("data", "hora", "ano", "mes", "dia", "hora_num").show(20)

In [ ]:
if "mes" in df_temporal.columns:
    df_temporal.groupBy("mes").agg(
        count("*").alias("total_registros"),
        round(avg("temperatura"), 2).alias("media_temperatura"),
        round(avg("umidade"), 2).alias("media_umidade"),
        round(avg("precipitacao"), 2).alias("media_precipitacao")
    ).orderBy("mes").show()

In [ ]:
if "mes" in df_temporal.columns:
    pdf_mes = (
        df_temporal
        .groupBy("mes")
        .agg(round(avg("temperatura"), 2).alias("media_temperatura"))
        .orderBy("mes")
        .toPandas()
    )

    plt.figure(figsize=(10, 5))
    plt.plot(pdf_mes["mes"], pdf_mes["media_temperatura"], marker="o")
    plt.title("Temperatura Média por Mês")
    plt.xlabel("Mês")
    plt.ylabel("Temperatura Média")
    plt.xticks(range(1, 13))
    plt.show()

In [ ]:
if "hora_num" in df_temporal.columns:
    df_temporal.groupBy("hora_num").agg(
        count("*").alias("total_registros"),
        round(avg("temperatura"), 2).alias("media_temperatura"),
        round(avg("umidade"), 2).alias("media_umidade"),
        round(avg("precipitacao"), 2).alias("media_precipitacao")
    ).orderBy("hora_num").show()

In [ ]:
if "hora_num" in df_temporal.columns:
    pdf_hora = (
        df_temporal
        .groupBy("hora_num")
        .agg(round(avg("temperatura"), 2).alias("media_temperatura"))
        .orderBy("hora_num")
        .toPandas()
    )

    plt.figure(figsize=(10, 5))
    plt.plot(pdf_hora["hora_num"], pdf_hora["media_temperatura"], marker="o")
    plt.title("Temperatura Média por Hora")
    plt.xlabel("Hora")
    plt.ylabel("Temperatura Média")
    plt.xticks(range(0, 24))
    plt.show()

In [ ]:
if "state" in df_temporal.columns:
    df_temporal.groupBy("state").agg(
        count("*").alias("total_registros"),
        round(avg("temperatura"), 2).alias("media_temperatura"),
        round(avg("umidade"), 2).alias("media_umidade"),
        round(avg("precipitacao"), 2).alias("media_precipitacao")
    ).orderBy("media_temperatura", ascending=False).show(truncate=False)

In [ ]:
if "state" in df_temporal.columns:
    pdf_estado = (
        df_temporal
        .groupBy("state")
        .agg(round(avg("temperatura"), 2).alias("media_temperatura"))
        .orderBy("media_temperatura", ascending=False)
        .toPandas()
    )

    plt.figure(figsize=(10, 5))
    plt.bar(pdf_estado["state"], pdf_estado["media_temperatura"])
    plt.title("Temperatura Média por Estado")
    plt.xlabel("Estado")
    plt.ylabel("Temperatura Média")
    plt.show()

In [ ]:
df_temporal.createOrReplaceTempView("weather")

In [ ]:
spark.sql("""
    SELECT 
        categoria_temperatura,
        COUNT(*) AS total_registros,
        ROUND(AVG(temperatura), 2) AS media_temperatura,
        ROUND(AVG(umidade), 2) AS media_umidade
    FROM weather
    GROUP BY categoria_temperatura
    ORDER BY media_temperatura
""").show(truncate=False)

In [ ]:
spark.sql("""
    SELECT 
        mes,
        COUNT(*) AS total_registros,
        ROUND(AVG(temperatura), 2) AS media_temperatura,
        ROUND(AVG(umidade), 2) AS media_umidade,
        ROUND(AVG(precipitacao), 2) AS media_precipitacao
    FROM weather
    WHERE mes IS NOT NULL
    GROUP BY mes
    ORDER BY mes
""").show(truncate=False)

In [ ]:
spark.sql("""
    SELECT 
        hora_num,
        COUNT(*) AS total_registros,
        ROUND(AVG(temperatura), 2) AS media_temperatura,
        ROUND(AVG(umidade), 2) AS media_umidade
    FROM weather
    WHERE hora_num IS NOT NULL
    GROUP BY hora_num
    ORDER BY hora_num
""").show(truncate=False)

In [ ]:
spark.sql("""
    SELECT 
        state,
        COUNT(*) AS total_registros,
        ROUND(AVG(temperatura), 2) AS media_temperatura,
        ROUND(AVG(umidade), 2) AS media_umidade,
        ROUND(AVG(precipitacao), 2) AS media_precipitacao
    FROM weather
    WHERE state IS NOT NULL
    GROUP BY state
    ORDER BY media_temperatura DESC
""").show(truncate=False)

In [ ]:
limite_correlacao = 0.05

features_selecionadas = (
    df_corr
    .filter(abs(col("correlacao_com_temperatura")) >= limite_correlacao)
    .select("variavel")
    .rdd.flatMap(lambda x: x)
    .collect()
)

print("Features selecionadas para modelagem:")
print(features_selecionadas)

In [ ]:
colunas_modelagem = features_selecionadas + ["temperatura"]

for c in ["mes", "hora_num"]:
    if c in df_temporal.columns and c not in colunas_modelagem:
        colunas_modelagem.append(c)

colunas_modelagem = [c for c in colunas_modelagem if c in df_temporal.columns]

df_modelagem = df_temporal.select(*colunas_modelagem).dropna()

df_modelagem.show(10)
df_modelagem.printSchema()

print(f"Total de linhas para modelagem: {df_modelagem.count()}")
print(f"Total de colunas para modelagem: {len(df_modelagem.columns)}")

In [ ]:
output_path = "/home/jovyan/work/data/processed/weather_eda_clean"

df_temporal.write.mode("overwrite").parquet(output_path)

print(f"Base tratada da EDA salva em: {output_path}")

In [ ]:
modelagem_path = "/home/jovyan/work/data/processed/weather_modelagem"

df_modelagem.write.mode("overwrite").parquet(modelagem_path)

print(f"Base otimizada para modelagem salva em: {modelagem_path}")